# ForgeLM — Data Curation: Ingestion + Audit

Two CLI commands turn raw enterprise corpora into training-ready data with a paper-trail an EU AI Act auditor will accept:

1. **`forgelm ingest`** — raw PDF / DOCX / EPUB / TXT / Markdown → SFT-ready JSONL. Optional PII + secrets masking, four chunking strategies, token-aware sizing.
2. **`forgelm audit`** — dataset quality + governance audit. Length distribution, language detection, near-duplicate scan (simhash or MinHash), cross-split leakage, PII severity tiers, secrets detection, quality heuristics.

**Requirements:** Any runtime works — no GPU needed. Both commands are CPU-bound and stream their input.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/data_curation.ipynb)

In [ ]:
# Step 1: Install ForgeLM with the ingestion extra (PDF/DOCX/EPUB/Markdown parsers + langdetect)
# Pinned to v0.5.2 — the release that introduced the markdown splitter,
# secrets tagger, MinHash dedup option, and quality filter exercised below.
!pip install -q --no-cache-dir 'forgelm[ingestion]==0.5.2'
!forgelm --version

## 1. Build a sample raw corpus

We simulate three realistic document types you might curate from — each
deliberately seeded with a near-duplicate, an embedded PII span, and a
credential leak. Real workflows would replace this step by pointing
`forgelm ingest` at a directory of policy PDFs / DOCX manuals.

In [ ]:
from pathlib import Path

raw = Path("./raw_docs")
raw.mkdir(exist_ok=True)

(raw / "handbook.md").write_text(
    """# Employee Handbook

Welcome to the company handbook. This document covers benefits and policies.

## Vacation

Salaried employees receive 25 paid vacation days per calendar year. Vacation
days do not roll over between calendar years and are forfeited if unused.

## Remote Work

Hybrid is the default. Two days in office, three days remote, with quarterly
in-person planning weeks. Contact HR (hr@example.com) for accommodations.

## Code of Conduct

All employees are expected to uphold professional standards in interactions
with colleagues and clients. Violations are reviewed by the ethics committee.
""",
    encoding="utf-8",
)

# A near-duplicate paragraph (only one word changed) — the audit should flag it.
(raw / "handbook_v2.md").write_text(
    """# Employee Handbook v2

Welcome to the company handbook. This document covers benefits and policies.

## Vacation

Salaried employees receive 25 paid vacation days per fiscal year. Vacation
days do not roll over between calendar years and are forfeited if unused.

## Office Locations

Headquarters in Berlin, satellite teams in Istanbul and Paris. Contact the
facilities team for visitor passes.
""",
    encoding="utf-8",
)

# Plain-text doc with embedded credential + PII.
# The credential-shaped substrings are *built at runtime* from inert
# fragments — the secret regexes still match the canonical shape, but
# repo-wide scanners (gitleaks / trufflehog / GitGuardian) don't see a
# full literal in the notebook source. Same convention as the test fixtures
# under tests/test_data_audit_phase12.py (FAKE_AWS_KEY / FAKE_GH_TOKEN).
FAKE_AWS_KEY = "AKIA" + "IOSFODNN7" + "EXAMPLE"
FAKE_GH_TOKEN = "ghp_" + "abcdefghij1234567890" + "ABCDEFGHIJ012345"

(raw / "deploy_runbook.txt").write_text(
    f"""DEPLOYMENT RUNBOOK

Operator: Alice (alice@example.com, +49 30 12345678)
Last reviewed: 2026-04-15.

Step 1: ssh into the bastion. The shared deploy key lives at /opt/deploy/id_rsa.
Step 2: export AWS_ACCESS_KEY_ID={FAKE_AWS_KEY} for the migration job.
Step 3: pull the latest container, run health checks, and rotate.

Emergency contact: ops@example.com.
GitHub deploy token: {FAKE_GH_TOKEN}.
""",
    encoding="utf-8",
)

for f in sorted(raw.iterdir()):
    print(f"  {f.name:<22} {f.stat().st_size:>5} bytes")

## 2. Ingest with the markdown-aware splitter (Phase 12)

`--strategy markdown` is the right choice when your corpus is structured
with headings: it splits at heading boundaries, keeps fenced code blocks
atomic, and **inlines the heading breadcrumb** at the top of every chunk
so SFT loss sees the document context. Falls back to paragraph mode for
non-markdown inputs (`.txt` here).

We also pass `--secrets-mask` and `--pii-mask` (Phase 12) — secrets run
**before** PII masking so combined detectors don't double-count
overlapping spans.

> **Phase 12.5 shorthand (`v0.5.3`):** `--all-mask` is equivalent to
> `--secrets-mask --pii-mask`. Use it when you want the
> "scrub-everything-detectable" workflow without remembering both
> flags. Composes additively with explicit flags (set-union).

In [ ]:
!forgelm ingest ./raw_docs/ \
    --recursive \
    --strategy markdown \
    --chunk-size 600 \
    --secrets-mask \
    --pii-mask \
    --output ./curated/policies.jsonl \
    --output-format json

In [ ]:
# Inspect the chunks. Note the heading breadcrumb prefixed onto every
# section's body, the redacted credential / PII spans, and that the
# .txt file fell back to the paragraph chunker (no headings to split on).
import json

with open("./curated/policies.jsonl", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        chunk = json.loads(line)["text"]
        print(f"=== Chunk {i} ({len(chunk)} chars) ===")
        print(chunk[:400] + ("…" if len(chunk) > 400 else ""))
        print()

## 3. Token-aware ingestion (Phase 11.5)

Character-mode chunks may exceed your model's token budget on dense
languages (Turkish, German). `--chunk-tokens N --tokenizer HF_ID` sizes
chunks against the actual vocabulary so they line up with `model.max_length`
exactly. `--tokenizer` is **required** with `--chunk-tokens` — we refuse
to default to a hidden vocab because the chunk count would silently
differ per-model.

(Skipping execution here — it would download a tokenizer; the command
below is the shape you'd run locally.)

In [ ]:
# forgelm ingest ./raw_docs/ \
#     --recursive \
#     --strategy markdown \
#     --chunk-tokens 1024 \
#     --tokenizer Qwen/Qwen2.5-7B-Instruct \
#     --output ./curated/policies_token_sized.jsonl
print("Run the above command locally if you want token-aware chunks.")

## 4. Audit the curated corpus (Phase 11.5 default + Phase 12 secrets)

`forgelm audit` produces a `data_audit_report.json` covering:

- per-split sample count, length distribution, top-3 detected languages
- LSH-banded simhash near-duplicate scan (default; `O(n × k)` typical case)
- cross-split leakage check
- PII severity tiers (`critical` / `high` / `medium` / `low` + `worst_tier`)
- **always-on secrets detection** — AWS / GitHub / Slack / OpenAI / Google /
  JWT / OpenSSH / PGP / Azure connection strings

If you ran `--secrets-mask` during ingest the secrets pass should now
show zero hits in the curated JSONL — confirming the masker did its job.

In [ ]:
!forgelm audit ./curated/policies.jsonl \
    --output ./audit/ \
    --output-format json

In [ ]:
# The on-disk report carries the full schema. Walk the highlights:
import json

with open("./audit/data_audit_report.json", encoding="utf-8") as fh:
    report = json.load(fh)

print("Total samples:", report.get("total_samples"))
print("Splits:       ", list(report.get("splits", {}).keys()))

near = report.get("near_duplicate_summary", {})
print(f"\nNear-duplicate method: {near.get('method')}")
print(f"  pairs per split:     {near.get('pairs_per_split')}")

pii = report.get("pii_summary", {})
sev = report.get("pii_severity", {})
if pii:
    print(f"\nPII summary: {pii}")
    print(f"  worst tier: {sev.get('worst_tier')}")

secrets = report.get("secrets_summary", {})
if secrets:
    print(f"\nSecrets summary (POST-MASK — should be empty if --secrets-mask did its job): {secrets}")
else:
    print("\nSecrets summary: clean (no credentials reached the JSONL).")

## 5. Opt-in heuristic quality filter (Phase 12)

`--quality-filter` adds a `quality_summary` block with five Gopher / C4
/ RefinedWeb-style heuristics. None of them block training; they surface
in the audit so the operator decides whether to drop / re-collect:

- `low_alpha_ratio` — < 70 % of non-whitespace chars are letters
- `low_punct_endings` — < 50 % of non-empty lines end with punctuation
- `abnormal_mean_word_length` — outside the 3–12 char window
- `short_paragraphs` — > 50 % of `\n\n`-blocks have < 5 words
- `repeated_lines` — top-3 actually-repeating lines covering > 30 %
  (catches boilerplate headers / footers / disclaimers)

Fenced markdown code blocks are stripped before applying the heuristics
so legitimate code-instruction rows don't trip prose-oriented checks.

In [ ]:
!forgelm audit ./curated/policies.jsonl \
    --output ./audit_quality/ \
    --quality-filter

In [ ]:
import json

with open("./audit_quality/data_audit_report.json", encoding="utf-8") as fh:
    quality_report = json.load(fh)

qs = quality_report.get("quality_summary", {})
if qs:
    print(f"Samples flagged: {qs.get('samples_flagged')}")
    print(f"Overall quality score: {qs.get('overall_quality_score')}")
    print("By check:")
    for check, n in (qs.get("by_check") or {}).items():
        print(f"  {check}: {n}")
else:
    print("No quality issues flagged — all rows passed the heuristics.")

## 6. MinHash LSH at corpus scale (Phase 12, opt-in)

For corpora above ~50K rows, simhash + LSH banding starts to feel its
edge: the band-bucket fan-out grows and false-positive checks dominate
the wall clock. Phase 12 adds an opt-in **MinHash LSH** path via the
optional `[ingestion-scale]` extra (the `datasketch` package).

MinHash measures **set-Jaccard** similarity over distinct tokens, while
the default simhash measures **frequency-weighted bit-cosine** — the two
are not interchangeable on identical thresholds, but in practice
simhash Hamming ≤ 3 ≈ MinHash Jaccard ≥ 0.85. Pin `num_perm` for
cross-run determinism.

(The CLI invocation below is gated on the optional extra; install if
you want to run it.)

In [ ]:
# Install the optional [ingestion-scale] extra so the install hint baked
# into ``forgelm.data_audit._require_datasketch`` matches the recipe used
# here. This pulls ``datasketch>=1.6.0,<2.0.0`` along with any future
# scale-tier deps the extra absorbs.
!pip install -q 'forgelm[ingestion-scale]==0.5.2'
!forgelm audit ./curated/policies.jsonl \
    --output ./audit_minhash/ \
    --dedup-method minhash \
    --jaccard-threshold 0.85

In [ ]:
import json

with open("./audit_minhash/data_audit_report.json", encoding="utf-8") as fh:
    mh_report = json.load(fh)

near = mh_report.get("near_duplicate_summary", {})
print(f"Method:               {near.get('method')}")
print(f"Threshold (Jaccard):  {near.get('jaccard_threshold')}")
print(f"num_perm (pin this):  {near.get('num_perm')}")
print(f"Pairs per split:      {near.get('pairs_per_split')}")

## 7. EU AI Act Article 10 governance integration

When `data_audit_report.json` is present in the trainer's `output_dir`
*before* training starts, the compliance bundle (Article 10 governance
artifact) auto-inlines its content. **Run the audit first, then train**
— the bundle stays self-contained and a deployer can verify the data
characteristics without rerunning anything.

```bash
# Recommended pre-training flow:
forgelm ingest ./raw_docs/ --recursive --secrets-mask --pii-mask \
    --output ./checkpoints/data/policies.jsonl

forgelm audit ./checkpoints/data/policies.jsonl \
    --output ./checkpoints/  # report lands next to the upcoming model

forgelm --config job.yaml      # training reads & inlines the audit
```

Stop-the-line: any non-zero `secrets_summary` count in the audit means
credentials reached the JSONL — fix the corpus / re-run with
`--secrets-mask` before training, otherwise the model will memorise
them.

## What's next

- **Train on the curated JSONL**: point any of the training notebooks at
  `./curated/policies.jsonl` via `data.dataset_name_or_path`.
  - [`quickstart_sft.ipynb`](./quickstart_sft.ipynb) — first SFT run
  - [`multi_dataset.ipynb`](./multi_dataset.ipynb) — mix the curated
    set with public datasets
- **Long-form guides**:
  - [`docs/guides/ingestion.md`](https://github.com/cemililik/ForgeLM/blob/main/docs/guides/ingestion.md)
    — every flag, every chunking strategy, every extra
  - [`docs/guides/data_audit.md`](https://github.com/cemililik/ForgeLM/blob/main/docs/guides/data_audit.md)
    — audit JSON schema, severity tiers, governance integration
- **Wizard**: `forgelm --wizard` offers to run `forgelm ingest` inline
  when you point it at a directory of raw docs (Phase 11.5+), and as of
  Phase 12.5 it also offers to run `forgelm audit` automatically once a
  JSONL is selected — closes the BYOD audit loop end-to-end.
- **Phase 12.5 audit add-ons (v0.5.3)**:
  - `--croissant` emits a [Google Croissant 1.0](http://mlcommons.org/croissant/)
    dataset card under the report's `croissant` key so the same JSON
    file doubles as both the EU AI Act Article 10 governance artifact
    and a Croissant-consumer dataset card.
  - `--pii-ml [--pii-ml-language LANG]` layers
    [Presidio](https://github.com/microsoft/presidio) NER on top of
    the regex detector — adds `person` / `organization` /
    `location` categories to `pii_summary` / `pii_severity`. **Two-step
    install:**
    ```bash
    pip install 'forgelm[ingestion-pii-ml]'
    python -m spacy download en_core_web_lg
    ```
    `presidio-analyzer` does not transitively ship a spaCy NER model;
    without the model `--pii-ml` raises an `ImportError` with the
    install recipe **before** any rows are scanned. Pass
    `--pii-ml-language tr` (and download the matching spaCy model) to
    audit Turkish corpora.